# Notebook 01 — ETL Pipeline — Student Version

## Goal

Build a **versioned, privacy-aware ML feature dataset** from the QBC12 Airbnb PostgreSQL database.

Final output:

- one row per `listing_id`
- one fixed `cutoff_date`
- features built only from data available before/on the cutoff
- target built from future calendar availability
- no raw PII columns in the final ML dataset

The next notebook will use this output for MLflow experiments. If this ETL is messy, the ML notebook will be garbage.

## What you must submit from this notebook

By the end, your notebook must save these files under `data/features/`:

```text
listing_availability_features_<version>.csv
listing_availability_features_<version>.parquet
listing_availability_features_<version>_metadata.json
listing_availability_features_<version>_validation_report.json
pii_audit_<version>.csv
```

The notebook must also show:

1. database connection check,
2. table/column inspection,
3. PII audit,
4. cutoff-date logic,
5. feature construction,
6. label construction,
7. validation checks.

## 0. Imports

These libraries are enough for the ETL notebook.

Install missing packages with:

```bash
pip install pandas numpy sqlalchemy psycopg2-binary pyarrow
```

In [ ]:
import os
import json
import re
from pathlib import Path
from datetime import timedelta

import numpy as np
import pandas as pd

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

## 1. Configuration

These values define the dataset version and the time windows.

- `PAST_WINDOW_DAYS`: how much history is used for features.
- `FUTURE_WINDOW_DAYS`: how much future data is used for the target.
- `HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD`: the rule for the positive class.

If you change any of these, change `DATASET_VERSION`.

In [ ]:
# -----------------------------
# ETL Configuration
# -----------------------------
DATASET_VERSION = "v1_student"

ENTITY_COLUMN = "listing_id"

PAST_WINDOW_DAYS = 90
FUTURE_WINDOW_DAYS = 30
HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD = 0.30

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
FEATURE_DIR = DATA_DIR / "features"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FEATURE_DIR:", FEATURE_DIR)

## 2. Database connection

Use your assigned student database user.

The QBC12 database is:

host: 185.50.38.163

port: 32112

database: qbc12_airbnb

Important:

- Keep `sslmode=disable`.
- Do not commit real passwords to Git.

In [ ]:
# -----------------------------
# Database Connection
# -----------------------------
import os
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
import pandas as pd

# Load .env file from the current project folder if present.
load_dotenv(find_dotenv(usecwd=True))

def load_windows_bat_env(path="enviroment.bat"):
    """Load simple `set KEY=value` assignments from the homework batch file."""
    if not os.path.exists(path):
        return
    with open(path, "r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line.lower().startswith("set ") or "=" not in line:
                continue
            key, value = line[4:].split("=", 1)
            key, value = key.strip(), value.strip()
            if key and value and not os.getenv(key):
                os.environ[key] = value

load_windows_bat_env()

# Support both DB_* names from enviroment.bat/.env and standard PG* names.
DB_HOST = os.getenv("DB_HOST") or os.getenv("PGHOST") or "185.50.38.163"
DB_PORT = int(os.getenv("DB_PORT") or os.getenv("PGPORT") or "32112")
DB_NAME = os.getenv("DB_NAME") or os.getenv("PGDATABASE") or "qbc12_airbnb"
DB_USER = os.getenv("DB_USER") or os.getenv("PGUSER")
DB_PASSWORD = os.getenv("DB_PASSWORD") or os.getenv("PGPASSWORD")

if not DB_USER or not DB_PASSWORD:
    raise ValueError(
        "Database credentials are missing. Set DB_USER and DB_PASSWORD in .env, "
        "enviroment.bat, or environment variables before running this notebook."
    )

print("Connecting to:")
print("HOST:", DB_HOST)
print("PORT:", DB_PORT)
print("DB:", DB_NAME)
print("USER:", DB_USER)
print("PASSWORD loaded:", bool(DB_PASSWORD))

db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
    query={"sslmode": "disable"},
)

engine = create_engine(db_url)

def read_sql(query: str, params: dict | None = None) -> pd.DataFrame:
    """Run a SQL query through SQLAlchemy and return a Pandas DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn, params=params)

with engine.connect() as conn:
    connection_check = conn.execute(
        text("""
        SELECT
            current_database() AS database,
            current_user AS user_name,
            inet_server_addr() AS server_ip,
            inet_server_port() AS server_port,
            now() AS checked_at;
        """)
    ).mappings().first()

dict(connection_check)


## 3. Inspect the available data

Before writing ETL, inspect the database.

You should confirm:

- which tables exist,
- which columns exist,
- how many rows each table has,
- whether important fields are missing.

In [ ]:
tables_df = read_sql("""
SELECT
    table_schema,
    table_name,
    table_type
FROM information_schema.tables
WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
ORDER BY table_schema, table_name;
""")

tables_df

In [ ]:
columns_df = read_sql("""
SELECT
    table_schema,
    table_name,
    ordinal_position,
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'core'
ORDER BY table_schema, table_name, ordinal_position;
""")

columns_df

In [ ]:
row_counts_df = read_sql("""
SELECT 'core.calendar_day' AS table_name, COUNT(*) AS row_count FROM core.calendar_day
UNION ALL
SELECT 'core.host' AS table_name, COUNT(*) AS row_count FROM core.host
UNION ALL
SELECT 'core.listing' AS table_name, COUNT(*) AS row_count FROM core.listing
UNION ALL
SELECT 'core.neighbourhood' AS table_name, COUNT(*) AS row_count FROM core.neighbourhood
UNION ALL
SELECT 'core.review' AS table_name, COUNT(*) AS row_count FROM core.review
ORDER BY table_name;
""")

row_counts_df

## 4. Data quality audit

This step decides which columns are safe and useful.

You must check at least:

1. calendar date range,
2. review date range,
3. whether `calendar_day.price` and `adjusted_price` are usable,
4. whether recent review windows are meaningful.

Do not include columns that are all-null or nearly useless.

In [ ]:
calendar_quality_df = read_sql("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE price IS NULL) AS null_price,
    COUNT(*) FILTER (WHERE adjusted_price IS NULL) AS null_adjusted_price,
    COUNT(*) FILTER (WHERE available IS NULL) AS null_available,
    MIN(date) AS min_calendar_date,
    MAX(date) AS max_calendar_date
FROM core.calendar_day;
""")

review_quality_df = read_sql("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE comment_len IS NULL) AS null_comment_len,
    MIN(review_date) AS min_review_date,
    MAX(review_date) AS max_review_date
FROM core.review;
""")

display(calendar_quality_df)
display(review_quality_df)

In [ ]:
# Inspect small samples.
# Keep LIMIT small. Do not pull full raw calendar/review tables into Pandas.

for table_name in ["listing", "host", "neighbourhood", "review", "calendar_day"]:
    print(f"\n===== core.{table_name} =====")
    display(read_sql(f"SELECT * FROM core.{table_name} LIMIT 10;"))

In [ ]:
columns_df = read_sql("""
SELECT
    table_name,
    ordinal_position,
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'core'
  AND table_name = 'listing'
ORDER BY ordinal_position;
""")

columns_df

## 5. Choose the cutoff date

The cutoff separates features from the label.

Rules:

- Historical features use dates `history_start_date <= date <= cutoff_date`.
- The label uses dates `cutoff_date < date <= label_end_date`.
- The cutoff must have enough past calendar data and enough future calendar data.

For this homework, use a 90-day feature window and a 30-day label window.

In [ ]:
range_df = read_sql("""
SELECT
    (SELECT MIN(date) FROM core.calendar_day) AS calendar_min_date,
    (SELECT MAX(date) FROM core.calendar_day) AS calendar_max_date,
    (SELECT MIN(review_date) FROM core.review) AS review_min_date,
    (SELECT MAX(review_date) FROM core.review) AS review_max_date;
""")

range_df

In [ ]:
calendar_min_date = pd.to_datetime(range_df.loc[0, "calendar_min_date"]).date()
calendar_max_date = pd.to_datetime(range_df.loc[0, "calendar_max_date"]).date()

# 1. earliest valid cutoff
earliest_cutoff_allowed_by_calendar = calendar_min_date + timedelta(days=PAST_WINDOW_DAYS)
# 2. latest valid cutoff
latest_cutoff_allowed_by_calendar = calendar_max_date - timedelta(days=FUTURE_WINDOW_DAYS)

# 3. choose a cutoff — وسط بازه‌ی مجاز انتخاب امن و قابل تکراریه
cutoff_date = latest_cutoff_allowed_by_calendar   # نزدیک‌ترین cutoff که آینده‌ی کافی داره

# 4. compute history & label windows
history_start_date = cutoff_date - timedelta(days=PAST_WINDOW_DAYS)
label_end_date = cutoff_date + timedelta(days=FUTURE_WINDOW_DAYS)

print("calendar_min_date:", calendar_min_date)
print("calendar_max_date:", calendar_max_date)
print("earliest_cutoff_allowed:", earliest_cutoff_allowed_by_calendar)
print("latest_cutoff_allowed:", latest_cutoff_allowed_by_calendar)
print("cutoff_date:", cutoff_date)
print("history_start_date:", history_start_date)
print("label_end_date:", label_end_date)

assert cutoff_date >= earliest_cutoff_allowed_by_calendar, "Not enough past calendar data."
assert cutoff_date <= latest_cutoff_allowed_by_calendar, "Not enough future calendar data."
assert history_start_date >= calendar_min_date
assert label_end_date <= calendar_max_date
print("\nCutoff windows are valid ✅")

## 6. PII audit

Raw identifiers can be needed for joins, but they must not become model features.

Your final ML feature table must not contain:

- `host_id`
- `host_pseudo_id`
- `review_id`
- `reviewer_id`
- `reviewer_pseudo_id`
- `license`
- raw text fields that may contain sensitive information

`listing_id` may stay as an entity key, but it must be excluded from model inputs later.

In [ ]:
# TODO: complete the PII audit table.
# Add rows for all sensitive or identity-linking columns you find relevant.

pii_audit = pd.DataFrame([
    {
        "table": "listing",
        "column": "listing_id",
        "pii_type": "entity identifier",
        "decision": "keep as entity key only",
        "reason": "needed to define one row per listing; not a model input"
    },
    # TODO: add host_id
    # TODO: add host_pseudo_id
    # TODO: add license
    # TODO: add review_id
    # TODO: add reviewer_id
    # TODO: add reviewer_pseudo_id
])

pii_audit

In [ ]:
pii_audit = pd.DataFrame([
    {"table": "listing", "column": "listing_id", "pii_type": "entity identifier",
     "decision": "keep as entity key only",
     "reason": "needed to define one row per listing; not a model input"},
    {"table": "listing", "column": "host_id", "pii_type": "direct identifier (host)",
     "decision": "drop from features (use only for join)",
     "reason": "links listing to a real person; not allowed as model input"},
    {"table": "host", "column": "host_pseudo_id", "pii_type": "pseudonymous identifier",
     "decision": "drop from features (join only)",
     "reason": "still re-identifies a host across listings"},
    {"table": "listing", "column": "license", "pii_type": "regulatory/legal identifier",
     "decision": "drop",
     "reason": "may contain government license numbers; sensitive & high-missing"},
    {"table": "review", "column": "review_id", "pii_type": "record identifier",
     "decision": "drop (aggregate only)",
     "reason": "row-level identifier, not a meaningful feature"},
    {"table": "review", "column": "reviewer_id", "pii_type": "direct identifier (reviewer)",
     "decision": "drop (aggregate to counts only)",
     "reason": "identifies a real reviewer"},
    {"table": "review", "column": "reviewer_pseudo_id", "pii_type": "pseudonymous identifier",
     "decision": "drop (count distinct only)",
     "reason": "re-identifies reviewers across reviews"},
])

pii_audit

## 7. Extract static tables

`listing`, `host`, and `neighbourhood` are small enough to load directly.

Do not load full `review` or `calendar_day` into Pandas. Those must be aggregated in SQL later.

In [ ]:
def get_table_columns(table_name: str) -> list[str]:
    df = read_sql("""
    SELECT column_name
    FROM information_schema.columns
    WHERE table_schema = 'core'
      AND table_name = :table_name
    ORDER BY ordinal_position;
    """, params={"table_name": table_name})
    return df["column_name"].tolist()

In [ ]:
listing_df = read_sql("""
SELECT
    listing_id,
    host_id,
    neighbourhood_id,
    room_type,
    property_type,
    accommodates,
    bedrooms,
    beds,
    bathrooms_text,
    listing_price,
    minimum_nights,
    maximum_nights,
    instant_bookable,
    license
FROM core.listing;
""")

# -----------------------------
# Host table: select only existing columns
# -----------------------------
host_cols = get_table_columns("host")
desired_host_cols = [
    "host_id",
    "host_pseudo_id",
    "is_superhost",
    "host_response_rate",
    "host_acceptance_rate",
]

existing_host_cols = [c for c in desired_host_cols if c in host_cols]

if "host_id" not in existing_host_cols:
    raise ValueError("host_id does not exist in core.host, cannot join host table.")

host_query = f"""
SELECT
    {", ".join(existing_host_cols)}
FROM core.host;
"""

host_df = read_sql(host_query)


# -----------------------------
# Neighbourhood table: select only existing columns
# -----------------------------
neighbourhood_cols = get_table_columns("neighbourhood")

neighbourhood_select_parts = []

if "neighbourhood_id" in neighbourhood_cols:
    neighbourhood_select_parts.append("neighbourhood_id")
else:
    raise ValueError("neighbourhood_id does not exist in core.neighbourhood.")

if "name" in neighbourhood_cols:
    neighbourhood_select_parts.append("name AS neighbourhood_name")

if "neighbourhood_group" in neighbourhood_cols:
    neighbourhood_select_parts.append("neighbourhood_group")

neighbourhood_query = f"""
SELECT
    {", ".join(neighbourhood_select_parts)}
FROM core.neighbourhood;
"""

neighbourhood_df = read_sql(neighbourhood_query)

print("listing:", listing_df.shape)
print("host:", host_df.shape, host_df.columns.tolist())
print("neighbourhood:", neighbourhood_df.shape, neighbourhood_df.columns.tolist())

## 8. Clean static fields

Convert database values into ML-friendly columns.

Required work:

- convert booleans to boolean dtype,
- convert numeric listing columns to numeric dtype,
- parse `bathrooms_text` into a numeric `bathrooms` feature.

In [ ]:
def to_bool(s):
    return (
        s.astype(str).str.strip().str.lower()
        .map({"t": True, "true": True, "1": True, "yes": True,
              "f": False, "false": False, "0": False, "no": False})
        .astype("boolean")
    )

# instant_bookable خودش boolean بود (طبق اسکیما)
if "instant_bookable" in listing_df.columns and listing_df["instant_bookable"].dtype not in ("boolean", bool):
    listing_df["instant_bookable"] = to_bool(listing_df["instant_bookable"])

# is_superhost: ممکنه bool یا text باشه — امن نرمالایز می‌کنیم
if "is_superhost" in host_df.columns and host_df["is_superhost"].dtype not in ("boolean", bool):
    host_df["is_superhost"] = to_bool(host_df["is_superhost"])

numeric_cols_listing = ["accommodates", "bedrooms", "beds",
                        "listing_price", "minimum_nights", "maximum_nights"]
for col in numeric_cols_listing:
    if col in listing_df.columns:
        listing_df[col] = pd.to_numeric(listing_df[col], errors="coerce")


def parse_bathrooms(text_value):
    if text_value is None or (isinstance(text_value, float) and np.isnan(text_value)):
        return np.nan
    s = str(text_value).strip().lower()
    if s in ("", "nan"):
        return np.nan
    if "half" in s:
        return 0.5
    m = re.search(r"(\d+(\.\d+)?)", s)
    return float(m.group(1)) if m else np.nan


listing_df["bathrooms"] = listing_df["bathrooms_text"].apply(parse_bathrooms)
listing_df[["bathrooms_text", "bathrooms"]].head(10)

## 9. Build static listing features

Join:

- `listing` → `host`
- `listing` → `neighbourhood`
- host-level aggregate `host_listing_count`

Final static features should be one row per `listing_id`.

Do not keep raw `host_id`, `host_pseudo_id`, `neighbourhood_id`, `license`, or `bathrooms_text` in the final static feature table.

In [ ]:
host_listing_features = (
    listing_df.groupby("host_id")["listing_id"]
    .nunique()
    .reset_index(name="host_listing_count")
)

base_listing_features = (
    listing_df
    .merge(host_df, on="host_id", how="left")
    .merge(host_listing_features, on="host_id", how="left")
    .merge(neighbourhood_df, on="neighbourhood_id", how="left")
)

# فقط فیچرهایی که واقعاً موجودن — host_response_rate / acceptance_rate / neighbourhood_group حذف می‌شن
static_feature_cols = [
    "listing_id",
    "room_type",
    "property_type",
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
    "listing_price",
    "minimum_nights",
    "maximum_nights",
    "instant_bookable",
    "is_superhost",
    "host_response_rate",       # وجود نداره → فیلتر می‌شه
    "host_acceptance_rate",     # وجود نداره → فیلتر می‌شه
    "host_listing_count",
    "neighbourhood_name",
    "neighbourhood_group",      # وجود نداره → فیلتر می‌شه
]
static_feature_cols = [c for c in static_feature_cols if c in base_listing_features.columns]

static_features = base_listing_features[static_feature_cols].copy()
assert static_features["listing_id"].duplicated().sum() == 0
print(static_features.shape)
print("Selected columns:", static_feature_cols)
static_features.head()

## 10. Build review features in SQL

Do not load raw `core.review` into Pandas.

Build one row per listing in SQL.

Required output columns:

- `listing_id`
- `total_reviews_before_cutoff`
- `unique_reviewers_before_cutoff`
- `avg_comment_len_before_cutoff`
- `max_comment_len_before_cutoff`
- `days_since_last_review`

Use only reviews where `review_date <= cutoff_date`.

In [ ]:
tables_df = read_sql("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'core'
ORDER BY table_name;
""")
tables_df

In [ ]:
read_sql("""
SELECT table_name, ordinal_position, column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'core'
  AND table_name IN ('review', 'calendar', 'calendar_day', 'calendar_30')
ORDER BY table_name, ordinal_position;
""")

In [ ]:
review_features = read_sql(
    """
    SELECT
        listing_id,
        COUNT(*)                                    AS total_reviews_before_cutoff,
        COUNT(DISTINCT reviewer_id)                 AS unique_reviewers_before_cutoff,
        AVG(comment_len)                            AS avg_comment_len_before_cutoff,
        MAX(comment_len)                            AS max_comment_len_before_cutoff,
        (CAST(:cutoff_date AS date) - MAX(review_date)) AS days_since_last_review
    FROM core.review
    WHERE review_date <= CAST(:cutoff_date AS date)
    GROUP BY listing_id;
    """,
    params={"cutoff_date": cutoff_date},
)

num_cols = ["total_reviews_before_cutoff", "unique_reviewers_before_cutoff",
            "avg_comment_len_before_cutoff", "max_comment_len_before_cutoff",
            "days_since_last_review"]
for c in num_cols:
    review_features[c] = pd.to_numeric(review_features[c], errors="coerce")

assert review_features["listing_id"].duplicated().sum() == 0
print(review_features.shape)
review_features.head()

## 11. Build calendar history features in SQL

Do not load raw `core.calendar_day` into Pandas.

Build historical availability features using:

- 90-day history window,
- 30-day recent history window.

Do not include calendar price features unless your audit proves they are usable.

In [ ]:
calendar_features_all = read_sql(
    """
    SELECT
        listing_id,
        -- full history window (history_start .. cutoff)
        SUM(CASE WHEN available THEN 1 ELSE 0 END)            AS available_days_history,
        AVG(CASE WHEN available THEN 1.0 ELSE 0.0 END)        AS available_rate_history,
        AVG(price)                                            AS avg_price_history,
        AVG(minimum_nights)                                   AS avg_minimum_nights_calendar_history,
        AVG(maximum_nights)                                   AS avg_maximum_nights_calendar_history,
        COUNT(*)                                              AS calendar_days_observed_history,

        -- last 30 days before cutoff
        SUM(CASE WHEN date > CAST(:cutoff_date AS date) - INTERVAL '30 days'
                  AND available THEN 1 ELSE 0 END)            AS available_days_last_30d,
        AVG(CASE WHEN date > CAST(:cutoff_date AS date) - INTERVAL '30 days'
                 THEN (CASE WHEN available THEN 1.0 ELSE 0.0 END) END)
                                                              AS available_rate_last_30d,
        AVG(CASE WHEN date > CAST(:cutoff_date AS date) - INTERVAL '30 days'
                 THEN price END)                              AS avg_price_last_30d
    FROM core.calendar_day
    WHERE date >= CAST(:history_start_date AS date)
      AND date <= CAST(:cutoff_date AS date)
    GROUP BY listing_id;
    """,
    params={"history_start_date": history_start_date, "cutoff_date": cutoff_date},
)

for c in calendar_features_all.columns:
    if c != "listing_id":
        calendar_features_all[c] = pd.to_numeric(calendar_features_all[c], errors="coerce")

assert calendar_features_all["listing_id"].duplicated().sum() == 0
print(calendar_features_all.shape)
calendar_features_all.head()

## 12. Build the target label

The label is built from future calendar availability.

Positive class:

```text
high_demand_proxy = 1 if future_available_rate_30d <= 0.30
```

This is not confirmed booking demand. It is a low-availability proxy.

In [ ]:
label_df = read_sql(
    """
    SELECT
        listing_id,
        COUNT(*)                                       AS future_calendar_days_observed_30d,
        SUM(CASE WHEN available THEN 1 ELSE 0 END)      AS future_available_days_30d,
        AVG(CASE WHEN available THEN 1.0 ELSE 0.0 END)  AS future_available_rate_30d,
        CASE
            WHEN AVG(CASE WHEN available THEN 1.0 ELSE 0.0 END) <= :threshold
            THEN 1 ELSE 0
        END                                            AS high_demand_proxy
    FROM core.calendar_day
    WHERE date > CAST(:cutoff_date AS date)
      AND date <= CAST(:label_end_date AS date)
    GROUP BY listing_id;
    """,
    params={
        "cutoff_date": cutoff_date,
        "label_end_date": label_end_date,
        "threshold": HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD,
    },
)

for c in ["future_calendar_days_observed_30d", "future_available_days_30d",
          "future_available_rate_30d"]:
    label_df[c] = pd.to_numeric(label_df[c], errors="coerce")
label_df["high_demand_proxy"] = label_df["high_demand_proxy"].astype(int)

assert label_df["listing_id"].duplicated().sum() == 0
print(label_df.shape)
print(label_df["high_demand_proxy"].value_counts())
label_df.head()

In [ ]:
# Check label balance.

label_distribution = (
    label_df["high_demand_proxy"]
    .value_counts(dropna=False)
    .rename_axis("high_demand_proxy")
    .reset_index(name="count")
)

label_distribution["percentage"] = (
    label_distribution["count"] / label_distribution["count"].sum()
).round(4)

label_distribution

## 13. Join feature groups and label

Join all feature groups into one ML-ready table.

The final granularity must be:

```text
one row = one listing_id at one cutoff_date
```

Use an inner join with `label_df`, because rows without a target cannot be used for supervised learning.

In [ ]:
feature_df = (
    static_features
    .merge(review_features, on="listing_id", how="left")
    .merge(calendar_features_all, on="listing_id", how="left")
    .merge(label_df, on="listing_id", how="inner")   # فقط ردیف‌هایی که target دارن
)

feature_df["cutoff_date"] = pd.to_datetime(cutoff_date)
feature_df["dataset_version"] = DATASET_VERSION

# review counts: missing => 0
for c in ["total_reviews_before_cutoff", "unique_reviewers_before_cutoff"]:
    if c in feature_df.columns:
        feature_df[c] = feature_df[c].fillna(0)

# listingهای بدون review: days_since_last_review را با عدد بزرگ پر کن + flag
if "days_since_last_review" in feature_df.columns:
    big = (cutoff_date - calendar_min_date).days + PAST_WINDOW_DAYS
    feature_df["has_reviews"] = (feature_df["total_reviews_before_cutoff"] > 0).astype(int)
    feature_df["days_since_last_review"] = feature_df["days_since_last_review"].fillna(big)

assert feature_df["listing_id"].duplicated().sum() == 0
print(feature_df.shape)
feature_df.head()

## 14. Drop unusable columns

Before saving, remove bad feature columns.

Drop columns that are:

- more than 95% missing,
- constant across all rows,

but protect target/audit columns.

In [ ]:
protected_columns = {
    "listing_id", "cutoff_date", "dataset_version",
    "future_calendar_days_observed_30d", "future_available_days_30d",
    "future_available_rate_30d", "high_demand_proxy",
}
HIGH_MISSING_DROP_THRESHOLD = 0.95

missing_rates = feature_df.isna().mean()
high_missing = [c for c in feature_df.columns
                if c not in protected_columns and missing_rates[c] > HIGH_MISSING_DROP_THRESHOLD]

constant_cols = [c for c in feature_df.columns
                 if c not in protected_columns and feature_df[c].nunique(dropna=False) <= 1]

columns_to_drop = sorted(set(high_missing) | set(constant_cols))
print("Columns to drop:", columns_to_drop)

feature_df = feature_df.drop(columns=columns_to_drop)
print("New shape:", feature_df.shape)

## 15. Validate the final dataset

The validation step is mandatory.

Check:

1. no duplicate `listing_id + cutoff_date`,
2. target exists and is binary,
3. no missing target values,
4. no forbidden PII columns,
5. no future leakage columns in model inputs.

In [ ]:
target_and_future_cols = {
    "high_demand_proxy",
    "future_available_rate_30d",
    "future_available_days_30d",
    "future_calendar_days_observed_30d",
}
id_and_meta_cols = {"listing_id", "cutoff_date", "dataset_version"}

model_input_columns = [
    c for c in feature_df.columns
    if c not in target_and_future_cols and c not in id_and_meta_cols
]
print("Model input columns:", model_input_columns)

In [ ]:
duplicate_count = feature_df.duplicated(subset=["listing_id", "cutoff_date"]).sum()
missing_target_count = feature_df["high_demand_proxy"].isna().sum()
unique_target_values = sorted(feature_df["high_demand_proxy"].dropna().unique().tolist())

forbidden_columns = {
    "host_id",
    "host_pseudo_id",
    "reviewer_id",
    "reviewer_pseudo_id",
    "review_id",
    "license",
    "bathrooms_text",
}

present_forbidden_columns = sorted(forbidden_columns.intersection(feature_df.columns))

label_only_columns = [
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
]

model_input_columns = [
    col for col in feature_df.columns
    if col not in label_only_columns
    and col not in ["listing_id", "cutoff_date", "dataset_version"]
]

future_leakage_columns = [
    col for col in model_input_columns
    if col.startswith("future_")
]

assert duplicate_count == 0, "Duplicate listing_id + cutoff_date rows found."
assert missing_target_count == 0, "Missing target values found."
assert set(unique_target_values).issubset({0, 1}), "Target must be binary 0/1."
assert not present_forbidden_columns, f"Forbidden PII columns present: {present_forbidden_columns}"
assert not future_leakage_columns, f"Future leakage columns in model inputs: {future_leakage_columns}"

print("duplicate_count:", duplicate_count)
print("missing_target_count:", missing_target_count)
print("unique_target_values:", unique_target_values)
print("present_forbidden_columns:", present_forbidden_columns)
print("future_leakage_columns:", future_leakage_columns)
print("model_input_column_count:", len(model_input_columns))


In [ ]:
missing_report = (
    feature_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

missing_report.columns = ["column", "missing_rate"]

calendar_coverage_summary = (
    feature_df[["future_calendar_days_observed_30d"]]
    .describe()
    .reset_index()
)

display(missing_report.head(30))
display(label_distribution)
display(calendar_coverage_summary)

## 16. Save versioned outputs

Save:

- feature dataset,
- metadata,
- validation report,
- PII audit.

The MLflow notebook must read this output instead of querying raw database tables again.

In [ ]:
csv_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.csv"
parquet_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.parquet"
metadata_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_metadata.json"
validation_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_validation_report.json"
pii_audit_path = FEATURE_DIR / f"pii_audit_{DATASET_VERSION}.csv"

feature_df.to_csv(csv_path, index=False)
print("Saved CSV:", csv_path)

feature_df.to_parquet(parquet_path, index=False)
print("Saved Parquet:", parquet_path)

metadata = {
    "dataset_version": DATASET_VERSION,
    "entity_column": ENTITY_COLUMN,
    "cutoff_date": str(cutoff_date),
    "history_start_date": str(history_start_date),
    "label_end_date": str(label_end_date),
    "past_window_days": PAST_WINDOW_DAYS,
    "future_window_days": FUTURE_WINDOW_DAYS,
    "high_demand_available_rate_threshold": HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD,
    "source_tables": ["core.listing", "core.host", "core.neighbourhood", "core.review", "core.calendar_day"],
    "target_definition": "high_demand_proxy = 1 if future_available_rate_30d <= threshold, else 0",
    "excluded_pii_columns": sorted([
        "host_id", "host_pseudo_id", "review_id", "reviewer_id",
        "reviewer_pseudo_id", "license", "bathrooms_text",
    ]),
    "row_count": int(len(feature_df)),
    "column_count": int(feature_df.shape[1]),
    "model_input_columns": model_input_columns,
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

validation_report = {
    "duplicate_listing_cutoff_rows": int(duplicate_count),
    "missing_target_count": int(missing_target_count),
    "target_values": [int(v) for v in unique_target_values],
    "present_forbidden_columns": present_forbidden_columns,
    "future_leakage_columns_in_model_inputs": future_leakage_columns,
    "missing_report": missing_report.to_dict(orient="records"),
    "label_distribution": label_distribution.to_dict(orient="records"),
    "calendar_coverage_summary": calendar_coverage_summary.to_dict(orient="records"),
}

with open(validation_path, "w", encoding="utf-8") as f:
    json.dump(validation_report, f, indent=2, ensure_ascii=False)

pii_audit.to_csv(pii_audit_path, index=False)

print("Saved metadata:", metadata_path)
print("Saved validation report:", validation_path)
print("Saved PII audit:", pii_audit_path)


## 17. Final preview

Use this final cell to confirm the output shape and columns.

Before moving to Notebook 2, make sure:

- target column exists,
- model input columns do not include future columns,
- no forbidden PII columns are present,
- saved files exist in `data/features/`.

In [ ]:
print("Final shape:", feature_df.shape)

display(feature_df.head())

feature_df.info()